# Bengali Troll Meme Analysis — P3: Plain Prompt + Gold Target + Gold Mechanism (Qwen3.5-9B)

Bengali Multimodal Troll Meme Analysis
Setting P3 : Plain prompting + GOLD TARGET + GOLD TROLLING MECHANISM in the prompt
Model      : Qwen/Qwen3.5-9B  (natively multimodal Qwen, ~9B LM / ~10B total)
Platform   : Kaggle Notebook, GPU T4 x2 (fp16, sharded) or 1 x T4 (4-bit)
The model receives the meme image + the gold target category + the gold primary
trolling mechanism, and must produce ONLY an explanation.
No classification is evaluated here (both labels are given as input).
Evaluation (explanation vs. gold human explanation)
  BLEU-4 (sentence + corpus), ROUGE-1/2/L, METEOR, BERTScore P/R/F1, SBERT cosine,
  mean / std / 95% bootstrap CI, per gold-mechanism, per gold-target and per
  (target x mechanism) breakdowns, plus paired significance tests vs. P0/P1/P2.
Every "# %%" marker below is one Kaggle notebook cell.

## [Cell 1] Install dependencies (Kaggle: Settings -> Internet ON, Accelerator -> GPU T4 x2)

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers>=5.2", "accelerate>=1.1.0", "bitsandbytes",
                "scikit-learn", "scipy"], check=False)
# explanation metrics
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sacrebleu", "rouge-score", "bert-score", "sentence-transformers", "nltk"], check=False)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 4.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.1 which is incompatible.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.1 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 4.2 MB/s eta 0:00:00


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'sacrebleu', 'rouge-score', 'bert-score', 'sentence-transformers', 'nltk'], returncode=0)

## [Cell 2] Configuration  <-- EDIT THIS CELL ONLY

In [2]:
import os, glob, shutil

CFG = {
    # ---------------- model ----------------
    "MODEL_ID": "Qwen/Qwen3.5-9B",
    "LOAD_MODE": "fp16",          # "fp16" (2 x T4) | "4bit" (1 x T4) | "bf16" | "fp32" (CPU debug)

    # ---------------- data ----------------
    "DATA_FILE": "/kaggle/input/datasets/mranikmahmudshanto/trollmeme-bn/troll_only_text.json",
    "IMAGE_ROOT": "/kaggle/input/datasets/mranikmahmudshanto/trollmeme-bn/labeled_images/labeled_images",
    "CSV_COLS": {"id": "image", "target": "target_category", "mechanism": "trolling_mechanism",
                 "explanation": "explanation", "troll": "troll_label"},

    # ---------------- generation (keep identical to P0 for a fair comparison) ----------------
    "EXPLANATION_LANGUAGE": "English",
    "MAX_IMAGE_SIDE": 768,
    "MAX_NEW_TOKENS": 320,
    "DO_SAMPLE": False,
    "REPETITION_PENALTY": 1.05,
    "SEED": 42,
    "LIMIT": None,                # e.g. 30 for a timing test

    # ---------------- sharding / resume ----------------
    "SHARD": 0, "NUM_SHARDS": 1,  # e.g. NUM_SHARDS=2 and run two versions with SHARD=0 / SHARD=1
    # predictions from earlier versions (add the old notebook output via "Add Input")
    "RESUME_GLOB": "/kaggle/input/**/qwen35_9b_P3_gold_target_mechanism/predictions*.jsonl",

    # ---------------- earlier settings for the comparison (Cell 11) ----------------
    # Add each earlier notebook's output via "Add Input"; leave "" to skip a setting.
    "PREV_RUNS": {
        "P0-plain":         "/kaggle/input/**/qwen35_9b_P0_plain",
        "P1-gold-target":   "/kaggle/input/**/qwen35_9b_P1_gold_target",
        "P2-gold-mechanism": "/kaggle/input/**/qwen35_9b_P2_gold_mechanism",
    },

    # ---------------- explanation metrics ----------------
    "BERTSCORE_MODEL": "roberta-large",    # English explanations; "xlm-roberta-large" for Bengali
    "BERTSCORE_LANG": "en",
    "SBERT_MODEL": "sentence-transformers/all-mpnet-base-v2",

    # ---------------- output ----------------
    "SETTING": "P3-gold-target+mechanism",
    "RUN_NAME": "qwen35_9b_P3_gold_target_mechanism",
    "OUT_DIR": "/kaggle/working/results",
}
if not os.path.exists("/kaggle"):
    CFG["OUT_DIR"] = "./results"
RUN_DIR = os.path.join(CFG["OUT_DIR"], CFG["RUN_NAME"])
os.makedirs(RUN_DIR, exist_ok=True)
PRED_FILE = os.path.join(RUN_DIR, f"predictions_shard{CFG['SHARD']}.jsonl")

# copy predictions from previous versions so the run resumes instead of restarting
for f in glob.glob(CFG["RESUME_GLOB"], recursive=True):
    dst = os.path.join(RUN_DIR, os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)
        print("Resumed:", f)
print("Outputs ->", RUN_DIR)


Outputs -> /kaggle/working/results/qwen35_9b_P3_gold_target_mechanism


## [Cell 3] Label space and definitions (used to build the prompt and read gold labels)

In [3]:
TARGET_LABELS = ["individual", "organization", "community", "concept"]
MECH_LABELS = ["direct_derision", "comparative_stereotypical_ridicule",
               "ironic_deceptive_framing", "provocative_baiting"]
INVALID = "INVALID"          # used when the output cannot be mapped to a valid label

TARGET_DEFS = {
    "individual":   "a specific, identifiable person (e.g., a politician, celebrity, player, influencer).",
    "organization": "a formal institution or structured group (e.g., political party, government body, "
                    "company, university, team/club, media outlet, agency).",
    "community":    "people sharing an identity or affiliation (e.g., religion, gender, region, "
                    "nationality, profession, class, ideology, fandom).",
    "concept":      "a belief, behaviour, event, policy, practice, trend, or other abstract entity.",
}
MECH_DEFS = {
    "direct_derision": "explicit insult, humiliation, dismissal, or mockery of the target.",
    "comparative_stereotypical_ridicule": "degrading comparison, analogy, contrast, or stereotype.",
    "ironic_deceptive_framing": "sarcasm, ironic praise, parody, misleading framing, or "
                                "text-image contradiction.",
    "provocative_baiting": "inflammatory content intended to trigger conflict or outrage.",
}
TASKS = {
    "target":    {"labels": TARGET_LABELS, "gold": "gold_target",    "pred": "pred_target",
                  "title": "Target Identification"},
    "mechanism": {"labels": MECH_LABELS,   "gold": "gold_mechanism", "pred": "pred_mechanism",
                  "title": "Trolling Mechanism"},
}


## [Cell 4] P3 prompt: plain prompt + gold target + gold mechanism (no gold explanation)

In [4]:
TARGET_NAMES = {"individual": "Individual", "organization": "Organization",
                "community": "Community", "concept": "Concept"}
MECH_NAMES = {"direct_derision": "Direct Derision",
              "comparative_stereotypical_ridicule": "Comparative / Stereotypical Ridicule",
              "ironic_deceptive_framing": "Ironic / Deceptive Framing",
              "provocative_baiting": "Provocative Baiting"}


def build_prompt_p3(gold_target: str, gold_mechanism: str, lang: str = "English") -> str:
    return f"""You are an expert analyst of Bengali social-media troll memes.
The given image is a Bengali troll meme.

The target category and the primary trolling mechanism of this meme are given below. Treat them as correct information:
Target category: {TARGET_NAMES[gold_target]} - {TARGET_DEFS[gold_target]}
Trolling mechanism: {MECH_NAMES[gold_mechanism]} - {MECH_DEFS[gold_mechanism]}

Use the given labels, but ground your analysis in the visible Bengali text and visual content yourself.

Task - Explanation: write a concise, self-contained explanation in {lang} (3-5 sentences). Describe the relevant Bengali text and visual cues, the implied meaning, the specific {gold_target} that is targeted, and how the given mechanism is used to ridicule, humiliate, deceptively frame, stereotype, or provoke the target. Do not invent background facts that are not supported by the meme.

Return ONLY one valid JSON object, with no markdown and no text before or after it, in exactly this format:
{{"explanation": "<your explanation>"}}"""


print(build_prompt_p3("organization", "ironic_deceptive_framing", CFG["EXPLANATION_LANGUAGE"]))


You are an expert analyst of Bengali social-media troll memes.
The given image is a Bengali troll meme.

The target category and the primary trolling mechanism of this meme are given below. Treat them as correct information:
Target category: Organization - a formal institution or structured group (e.g., political party, government body, company, university, team/club, media outlet, agency).
Trolling mechanism: Ironic / Deceptive Framing - sarcasm, ironic praise, parody, misleading framing, or text-image contradiction.

Use the given labels, but ground your analysis in the visible Bengali text and visual content yourself.

Task - Explanation: write a concise, self-contained explanation in English (3-5 sentences). Describe the relevant Bengali text and visual cues, the implied meaning, the specific organization that is targeted, and how the given mechanism is used to ridicule, humiliate, deceptively frame, stereotype, or provoke the target. Do not invent background facts that are not sup

## [Cell 5] Output parser (explanation only; robust to <think> blocks, code fences, broken JSON)

In [5]:
import re, json, ast, difflib

TARGET_ALIASES = {
    "individual": ["individual", "person", "individual person", "specific person", "politician",
                   "celebrity", "public figure", "player", "influencer", "ind"],
    "organization": ["organization", "organisation", "institution", "political party", "party",
                     "company", "government", "university", "team", "club", "media", "agency", "org"],
    "community": ["community", "group", "social group", "religious group", "gender group", "fandom",
                  "fan group", "profession", "nationality", "regional group", "people", "com"],
    "concept": ["concept", "idea", "belief", "behaviour", "behavior", "event", "policy", "practice",
                "trend", "abstract", "abstract entity", "con"],
}
MECH_ALIASES = {
    "direct_derision": ["direct_derision", "direct derision", "derision", "direct", "insult",
                        "humiliation", "mockery", "ridicule", "drd"],
    "comparative_stereotypical_ridicule": ["comparative_stereotypical_ridicule",
                        "comparative/stereotypical ridicule", "comparative stereotypical ridicule",
                        "comparative ridicule", "stereotypical ridicule", "comparative",
                        "stereotypical", "stereotype", "comparison", "analogy", "csr"],
    "ironic_deceptive_framing": ["ironic_deceptive_framing", "ironic/deceptive framing",
                        "ironic deceptive framing", "ironic framing", "deceptive framing", "ironic",
                        "irony", "sarcasm", "sarcastic", "parody", "deceptive", "idf"],
    "provocative_baiting": ["provocative_baiting", "provocative baiting", "provocative", "baiting",
                        "bait", "provocation", "inflammatory", "rage bait", "pbt"],
}
ALIASES = {"target": TARGET_ALIASES, "mechanism": MECH_ALIASES}
KEYS = {  # accepted JSON keys per field
    "explanation": ["explanation", "explanation_text", "reasoning", "rationale"],
    "target": ["target_category", "target", "target_type", "target_identification"],
    "mechanism": ["trolling_mechanism", "mechanism", "primary_mechanism", "troll_mechanism"],
}


def _canon(s) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[\"'`*.,;:()\[\]{}]", " ", s)
    s = re.sub(r"[\s/\-]+", " ", s).strip()
    return s


def normalize_label(raw, task: str) -> str:
    """Map a raw string to a canonical label of `task` ('target' | 'mechanism') or INVALID."""
    if raw is None:
        return INVALID
    labels = TASKS[task]["labels"]
    s = _canon(raw)
    if not s or s in {"na", "n a", "none", "null", "unknown"}:
        return INVALID
    s_us = s.replace(" ", "_")
    if s_us in labels:
        return s_us
    alias_map = {_canon(a): lab for lab, al in ALIASES[task].items() for a in al}
    if s in alias_map:
        return alias_map[s]
    # the longest alias contained in the string (e.g. "political party (organization)")
    hits = sorted(((a, lab) for a, lab in alias_map.items()
                   if len(a) > 3 and re.search(rf"\b{re.escape(a)}\b", s)),
                  key=lambda x: -len(x[0]))
    if hits:
        best_len = len(hits[0][0])
        if len({lab for a, lab in hits if len(a) == best_len}) == 1:
            return hits[0][1]
    close = difflib.get_close_matches(s, list(alias_map), n=1, cutoff=0.80)
    return alias_map[close[0]] if close else INVALID


def strip_thinking(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S | re.I)
    if "</think>" in text:                      # opening tag lost in decoding
        text = text.split("</think>")[-1]
    return text.strip()


def _json_candidates(text: str):
    for m in re.finditer(r"```(?:json)?\s*(.*?)```", text, flags=re.S | re.I):
        yield m.group(1)
    depth, start = 0, None                      # balanced-brace scan
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}" and depth:
            depth -= 1
            if depth == 0 and start is not None:
                yield text[start:i + 1]
    if "{" in text and text.count("{") > text.count("}"):   # truncated object
        yield text[text.index("{"):] + '"}'


def _try_load(s: str):
    s = s.strip().replace("\u201c", '"').replace("\u201d", '"').replace("\u2019", "'")
    s2 = re.sub(r",\s*([}\]])", r"\1", s)       # trailing commas
    for cand in (s, s2):
        try:
            obj = json.loads(cand)
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass
        try:
            obj = ast.literal_eval(cand)
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass
    return None


def _get_key(d: dict, field: str):
    low = {str(k).strip().lower(): v for k, v in d.items()}
    for k in KEYS[field]:
        if k in low:
            return low[k]
    return None


def _regex_field(text: str, field: str):
    keys = "|".join(re.escape(k) for k in KEYS[field])
    if field == "explanation":
        m = re.search(rf'"?(?:{keys})"?\s*[:=]\s*"(.*?)"\s*(?:,\s*"|\}}|$)', text, flags=re.S | re.I)
        if not m:   # unquoted "Explanation: ..." style, up to the next field line
            m = re.search(rf'\**(?:{keys})\**\s*[:=]\s*(.+?)(?=\n\s*\**\s*(?:target|trolling|mechanism|primary)|\Z)',
                          text, flags=re.S | re.I)
    else:
        m = re.search(rf'"?(?:{keys})"?\s*[:=\-]\s*"?([A-Za-z_ /\-]+)', text, flags=re.I)
    return m.group(1).strip() if m else None


def _single_label_mention(text: str, task: str):
    """Last resort: accept a label only if exactly one canonical label is mentioned."""
    found = {lab for lab in TASKS[task]["labels"]
             if re.search(rf"\b{re.escape(lab)}\b|\b{re.escape(lab.replace('_', ' '))}\b", text, re.I)}
    return found.pop() if len(found) == 1 else None


def _clean_free_text(text: str) -> str:
    """Fallback when there is no JSON: drop code fences, braces and a leading 'Explanation:' tag."""
    t = re.sub(r"```(?:json)?", "", text, flags=re.I)
    t = re.sub(r'^\s*\{?\s*"?explanation"?\s*[:=]\s*"?', "", t, flags=re.I)
    t = re.sub(r'"?\s*\}?\s*$', "", t)
    return t.strip()


def parse_output(raw: str) -> dict:
    """Extract only the explanation. Also works on P0/P1/P2 outputs (extra keys are ignored)."""
    text = strip_thinking(raw or "")
    for cand in _json_candidates(text):
        obj = _try_load(cand)
        if obj is not None:
            val = _get_key(obj, "explanation")
            if isinstance(val, str) and val.strip():
                return {"pred_explanation": val.strip(), "parse_status": "json"}
    val = _regex_field(text, "explanation")
    if val:
        return {"pred_explanation": val.strip().rstrip('"}').strip(), "parse_status": "regex"}
    val = _clean_free_text(text)
    if val:
        return {"pred_explanation": val, "parse_status": "text"}
    return {"pred_explanation": "", "parse_status": "failed"}


# quick self-test of the parser
_tests = [
    '{"explanation": "The meme mocks a party by ironic praise."}',
    '<think>hmm</think>```json\n{"explanation":"Sarcastic caption about a cricketer.",}\n```',
    'Explanation: The meme compares the minister to a clown.',
    '{"explanation": "cut off mid sentence',
    '{"explanation": "x", "target_category": "organization", "trolling_mechanism": "sarcasm"}',  # P0-style
    '',
]
for t in _tests:
    print(parse_output(t))


{'pred_explanation': 'The meme mocks a party by ironic praise.', 'parse_status': 'json'}
{'pred_explanation': 'Sarcastic caption about a cricketer.', 'parse_status': 'json'}
{'pred_explanation': 'The meme compares the minister to a clown.', 'parse_status': 'regex'}
{'pred_explanation': 'cut off mid sentence', 'parse_status': 'json'}
{'pred_explanation': 'x', 'parse_status': 'json'}
{'pred_explanation': '', 'parse_status': 'failed'}


## [Cell 6] Load dataset (Chihna JSON export or CSV) and resolve image paths

In [6]:
import pandas as pd


def _index_images(root: str) -> dict:
    idx = {}
    exts = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif")
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if fn.lower().endswith(exts):
                idx.setdefault(fn, os.path.join(dp, fn))
                idx.setdefault(os.path.splitext(fn)[0], os.path.join(dp, fn))
    return idx


def _field(value, *keys):
    """value[key] if value is a dict, the value itself if it is a string/number, else None."""
    if value is None:
        return None
    if isinstance(value, dict):
        for k in keys:
            if value.get(k) not in (None, ""):
                return value[k]
        return None
    return value


def _first(a: dict, *names):
    for n in names:
        if a.get(n) not in (None, "", {}):
            return a[n]
    return None


def load_dataset(cfg) -> pd.DataFrame:
    path = cfg["DATA_FILE"]
    rows = []
    if path.lower().endswith(".json"):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        anns = data.get("annotations", data) if isinstance(data, dict) else data
        if isinstance(anns, dict):
            anns = [{**v, "image": v.get("image", k)} for k, v in anns.items()]
        for a in anns:
            if not isinstance(a, dict) or a.get("status") not in (None, "annotated"):
                continue
            troll = _field(_first(a, "troll_detection", "troll_label", "troll"), "label")
            if troll is not None and str(troll).strip().lower().replace("-", "_") not in ("troll", "1"):
                continue
            img = _first(a, "image", "filename", "image_path", "file")
            if isinstance(img, dict):
                rel = img.get("relative_path") or img.get("filename") or img.get("path")
                fname = img.get("filename") or (os.path.basename(rel) if rel else None)
            else:
                rel = str(img) if img is not None else None
                fname = os.path.basename(rel) if rel else None
            rows.append({
                "id": rel, "filename": fname,
                "gold_target": _field(_first(a, "target_identification", "target_category", "target"),
                                      "category", "label"),
                "gold_mechanism": _field(_first(a, "trolling_mechanism", "mechanism", "troll_mechanism"),
                                         "primary", "label"),
                "gold_explanation": _field(_first(a, "explanation", "gold_explanation", "final_explanation"),
                                           "text", "final_text") or "",
            })
        df = pd.DataFrame(rows, columns=["id", "filename", "gold_target", "gold_mechanism", "gold_explanation"])
    else:
        c = cfg["CSV_COLS"]
        raw = pd.read_csv(path)
        if c.get("troll") in raw.columns:
            raw = raw[raw[c["troll"]].astype(str).str.lower().str.replace("-", "_").isin(["troll", "1"])]
        df = pd.DataFrame({
            "id": raw[c["id"]].astype(str),
            "filename": raw[c["id"]].astype(str).map(os.path.basename),
            "gold_target": raw[c["target"]],
            "gold_mechanism": raw[c["mechanism"]],
            "gold_explanation": raw[c["explanation"]] if c["explanation"] in raw.columns else "",
        })
    if df.empty:
        raise ValueError("No records loaded - check DATA_FILE and the JSON field names.")

    df["gold_target"] = df["gold_target"].map(lambda x: normalize_label(x, "target"))
    df["gold_mechanism"] = df["gold_mechanism"].map(lambda x: normalize_label(x, "mechanism"))
    df["gold_explanation"] = df["gold_explanation"].fillna("").astype(str)
    bad = df[(df.gold_target == INVALID) | (df.gold_mechanism == INVALID)]
    if len(bad):
        print(f"[warn] dropping {len(bad)} rows with missing/unknown gold labels")
        df = df.drop(bad.index)

    idx = _index_images(cfg["IMAGE_ROOT"])

    def _resolve(r):
        p = os.path.join(cfg["IMAGE_ROOT"], str(r["id"]))
        if os.path.isfile(p):
            return p
        return idx.get(str(r["filename"])) or idx.get(os.path.splitext(str(r["filename"]))[0])

    df["image_path"] = df.apply(_resolve, axis=1)
    miss = df.image_path.isna().sum()
    if miss:
        print(f"[warn] {miss} images not found and skipped")
        df = df[df.image_path.notna()]
    df = df.drop_duplicates("id").reset_index(drop=True)
    if cfg["LIMIT"]:
        df = df.head(cfg["LIMIT"])
    return df


df_all = load_dataset(CFG)                                   # full set (used for evaluation)
df = df_all.iloc[CFG["SHARD"]::CFG["NUM_SHARDS"]].reset_index(drop=True)   # this shard (inference)
print(f"Loaded {len(df_all)} troll memes | this shard: {len(df)}")
print(df_all.gold_target.value_counts(), "\n")
print(df_all.gold_mechanism.value_counts())


Loaded 1200 troll memes | this shard: 1200
gold_target
individual      679
concept         256
community       240
organization     25
Name: count, dtype: int64 

gold_mechanism
direct_derision                       587
ironic_deceptive_framing              265
comparative_stereotypical_ridicule    235
provocative_baiting                   113
Name: count, dtype: int64


## [Cell 7] Load Qwen3.5-9B

In [7]:
import torch, random, numpy as np
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])
n_gpu = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(n_gpu)])

processor = AutoProcessor.from_pretrained(CFG["MODEL_ID"])
load_kw = {"device_map": "auto", "attn_implementation": "sdpa"}
if CFG["LOAD_MODE"] == "4bit":
    load_kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_skip_modules=["visual", "lm_head"],   # keep vision tower + head in fp16
    )
    load_kw["dtype"] = torch.float16
else:
    load_kw["dtype"] = {"bf16": torch.bfloat16, "fp32": torch.float32}.get(CFG["LOAD_MODE"], torch.float16)
    if n_gpu >= 2:   # leave head-room on GPU0 for the vision encoder + activations
        load_kw["max_memory"] = {0: "11GiB", **{i: "14GiB" for i in range(1, n_gpu)}}

model = AutoModelForImageTextToText.from_pretrained(CFG["MODEL_ID"], **load_kw).eval()
print("Params: %.2fB" % (sum(p.numel() for p in model.parameters()) / 1e9))
print(getattr(model, "hf_device_map", None) and set(model.hf_device_map.values()))


GPUs: ['Tesla T4', 'Tesla T4']


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/79.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Params: 9.41B
{0, 1}


## [Cell 8] Inference helpers

In [8]:
import time
from PIL import Image


def load_image(path: str, max_side: int) -> Image.Image:
    img = Image.open(path)
    if img.mode in ("RGBA", "LA", "P"):
        img = img.convert("RGBA")
        bg = Image.new("RGB", img.size, (255, 255, 255))
        bg.paste(img, mask=img.split()[-1])
        img = bg
    img = img.convert("RGB")
    w, h = img.size
    if max(w, h) > max_side:
        s = max_side / max(w, h)
        img = img.resize((max(28, int(w * s)), max(28, int(h * s))), Image.BICUBIC)
    return img


@torch.inference_mode()
def generate(image: Image.Image, prompt: str) -> str:
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)       # no <think> block
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
    gen_kw = dict(max_new_tokens=CFG["MAX_NEW_TOKENS"], do_sample=CFG["DO_SAMPLE"],
                  repetition_penalty=CFG["REPETITION_PENALTY"],
                  stop_strings=['"}', '" }', '"\n}'], tokenizer=processor.tokenizer)  # stop after the JSON
    if CFG["DO_SAMPLE"]:
        gen_kw.update(temperature=0.7, top_p=0.8, top_k=20)
    else:
        gen_kw.update(temperature=None, top_p=None, top_k=None)
    out = model.generate(**inputs, **gen_kw)
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()


def safe_generate(path: str, prompt: str) -> str:
    for side in (CFG["MAX_IMAGE_SIDE"], 640, 512):
        try:
            return generate(load_image(path, side), prompt)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"  OOM at {side}px, retrying smaller", flush=True)
    raise RuntimeError("OOM even at 512px")


# sanity check (skipped in "Save & Run All" commits)
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    print("Commit run: skipping sanity check")
else:
    _r = df.iloc[0]
    _t0 = time.time()
    _raw = safe_generate(_r.image_path, build_prompt_p3(_r.gold_target, _r.gold_mechanism,
                                                        CFG["EXPLANATION_LANGUAGE"]))
    print(f"({time.time() - _t0:.1f}s)\n{_raw}\n->", parse_output(_raw))
    if re.fullmatch(r"[!\s]*", _raw):
        raise RuntimeError("Degenerate output (NaN in fp16). Set LOAD_MODE='4bit' or 'bf16' and reload.")


Commit run: skipping sanity check


## [Cell 9] Run P3 over the dataset (resumable; one log line per meme)

In [9]:
def _done_ids() -> set:
    ids = set()
    for f in glob.glob(os.path.join(RUN_DIR, "predictions*.jsonl")):
        with open(f, encoding="utf-8") as fh:
            ids |= {json.loads(l)["id"] for l in fh if l.strip()}
    return ids


done = _done_ids()
todo = df[~df.id.isin(done)]
print(f"{len(df) - len(todo)} already done, {len(todo)} remaining in this shard", flush=True)

t_start = time.time()
with open(PRED_FILE, "a", encoding="utf-8") as fout:
    for k, (_, r) in enumerate(todo.iterrows(), 1):
        prompt = build_prompt_p3(r["gold_target"], r["gold_mechanism"], CFG["EXPLANATION_LANGUAGE"])
        t0 = time.time()
        try:
            raw, err = safe_generate(r["image_path"], prompt), None
        except Exception as e:
            raw, err = "", repr(e)
        parsed = parse_output(raw)
        rec = {"id": r["id"], "image_path": r["image_path"], "setting": CFG["SETTING"],
               "given_target": r["gold_target"], "given_mechanism": r["gold_mechanism"],
               "gold_target": r["gold_target"], "gold_mechanism": r["gold_mechanism"],
               "gold_explanation": r["gold_explanation"],
               "raw_output": raw, "error": err, "latency_s": round(time.time() - t0, 2),
               "pred_explanation": parsed["pred_explanation"],
               "parse_status": parsed["parse_status"]}
        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        fout.flush()
        eta_h = (time.time() - t_start) / k * (len(todo) - k) / 3600
        print(f"[{k}/{len(todo)}] {r['id']}  {rec['latency_s']}s  words={len(parsed['pred_explanation'].split())} "
              f"({rec['parse_status']})  ETA {eta_h:.1f}h" + (f"  ERR={err}" if err else ""), flush=True)


0 already done, 1200 remaining in this shard


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[1/1200] 92346762_533336734235946_3358438944999403328_n.jpg  17.02s  words=86 (json)  ETA 5.7h
[2/1200] 92358777_213685896577229_3055422575827376263_n.jpg  14.34s  words=96 (json)  ETA 5.2h
[3/1200] 92399719_1082342258796283_8342248266681858533_n.jpg  13.64s  words=93 (json)  ETA 5.0h
[4/1200] 92440695_231192111580290_6617336788896684503_n.jpg  17.98s  words=127 (json)  ETA 5.2h
[5/1200] 1.png  16.73s  words=105 (json)  ETA 5.3h
[6/1200] 1005.jpg  14.44s  words=95 (json)  ETA 5.2h
[7/1200] 1006.jpg  19.48s  words=126 (json)  ETA 5.4h
[8/1200] 1008.jpg  16.02s  words=104 (json)  ETA 5.4h
[9/1200] 1009.jpg  17.84s  words=112 (json)  ETA 5.4h
[10/1200] 1010.jpg  13.9s  words=81 (json)  ETA 5.3h
[11/1200] 1012.jpg  16.5s  words=114 (json)  ETA 5.3h
[12/1200] 1016.jpg  15.91s  words=107 (json)  ETA 5.3h
[13/1200] 1017.jpg  12.66s  words=83 (json)  ETA 5.2h
[14/1200] 102.jpg  15.08s  words=100 (json)  ETA 5.2h
[15/1200] 1020.jpg  13.29s  words=79 (json)  ETA 5.2h
[16/1200] 1022.jpg  15.1s  w

## [Cell 10] Explanation evaluation for P3 (reference = gold human explanation)

In [10]:
# Metrics: BLEU-4, ROUGE-1/2/L, METEOR, BERTScore P/R/F1, SBERT cosine similarity.
# To evaluate without re-running the model: run Cells 1-6, then Cells 10-12 (skip 7-9).
import gc, re as _re
import numpy as np
import torch
import sacrebleu, nltk
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bertscore
from sentence_transformers import SentenceTransformer


def load_predictions(run_dir: str, ids=None) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(run_dir, "predictions*.jsonl")))
    if not files:
        raise FileNotFoundError(f"No predictions*.jsonl in {run_dir}")
    p = pd.concat([pd.read_json(f, lines=True) for f in files], ignore_index=True)
    if ids is not None:
        p = p[p.id.isin(ids)]
    p = p.drop_duplicates("id", keep="last").reset_index(drop=True)
    # re-parse raw outputs so parser fixes apply without re-running the model
    parsed = p.raw_output.fillna("").map(parse_output).apply(pd.Series)
    p["pred_explanation"], p["parse_status"] = parsed["pred_explanation"], parsed["parse_status"]
    # gold columns always come from the current dataset file (same references for every setting)
    p = p.drop(columns=[c for c in ("gold_target", "gold_mechanism", "gold_explanation") if c in p])
    return p.merge(df_all[["id", "gold_target", "gold_mechanism", "gold_explanation"]], on="id", how="inner")


pred = load_predictions(RUN_DIR, ids=set(df_all.id))
print(f"Evaluating {len(pred)} / {len(df_all)} memes")
if len(pred) < len(df_all):
    print(f"[warn] {len(df_all) - len(pred)} memes have no prediction yet (other shard not finished?)")
print("Parse status:\n", pred.parse_status.value_counts().to_string())
if "error" in pred:
    print("Generation errors:", int(pred.error.notna().sum()))

# free the VLM if it is still loaded (explanation metrics need GPU memory)
if "model" in globals():
    del model
    gc.collect(); torch.cuda.empty_cache()

for pkg in ("wordnet", "omw-1.4"):
    nltk.download(pkg, quiet=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
_sbert = SentenceTransformer(CFG["SBERT_MODEL"], device=DEVICE)
_tok = lambda s: _re.findall(r"\w+|[^\w\s]", s.lower())


def _bootstrap_ci(x, n_boot=1000, seed=42):
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    means = rng.choice(x, size=(n_boot, len(x)), replace=True).mean(1)
    return np.percentile(means, 2.5), np.percentile(means, 97.5)


def evaluate_explanations(p: pd.DataFrame, setting: str, out_dir: str):
    hyps = p.pred_explanation.fillna("").astype(str).str.strip().tolist()
    refs = p.gold_explanation.fillna("").astype(str).str.strip().tolist()
    keep = [i for i, r in enumerate(refs) if r]                      # need a reference
    if len(keep) < len(refs):
        print(f"[warn] {len(refs) - len(keep)} items without gold explanation skipped")
    hyps, refs = [hyps[i] for i in keep], [refs[i] for i in keep]
    sub = p.iloc[keep].reset_index(drop=True)
    empty = np.array([h == "" for h in hyps])
    n = len(hyps)

    s = pd.DataFrame({"id": sub.id, "gold_target": sub.gold_target, "gold_mechanism": sub.gold_mechanism})
    s["bleu4"] = [sacrebleu.sentence_bleu(h, [r]).score / 100 if h else 0.0 for h, r in zip(hyps, refs)]
    rs = [_rouge.score(r, h) for h, r in zip(hyps, refs)]
    for k in ("rouge1", "rouge2", "rougeL"):
        s[k] = [x[k].fmeasure for x in rs]
    s["meteor"] = [meteor_score([_tok(r)], _tok(h)) if h else 0.0 for h, r in zip(hyps, refs)]

    # BERTScore (empty hypotheses get 0)
    ne = [i for i in range(n) if not empty[i]]
    bp = np.zeros(n); br = np.zeros(n); bf = np.zeros(n)
    rescaled = True
    if ne:
        kw = dict(model_type=CFG["BERTSCORE_MODEL"], lang=CFG["BERTSCORE_LANG"], batch_size=16,
                  device=DEVICE, verbose=False)
        try:
            P, R, F = bertscore([hyps[i] for i in ne], [refs[i] for i in ne], rescale_with_baseline=True, **kw)
        except Exception as e:                                        # no baseline for this model/lang
            print("[info] BERTScore without baseline rescaling:", e)
            rescaled = False
            P, R, F = bertscore([hyps[i] for i in ne], [refs[i] for i in ne], **kw)
        bp[ne], br[ne], bf[ne] = P.numpy(), R.numpy(), F.numpy()
    s["bertscore_p"], s["bertscore_r"], s["bertscore_f1"] = bp, br, bf

    # sentence-embedding cosine similarity
    eh = _sbert.encode(hyps, batch_size=32, normalize_embeddings=True, convert_to_numpy=True)
    er = _sbert.encode(refs, batch_size=32, normalize_embeddings=True, convert_to_numpy=True)
    s["sbert_cosine"] = np.where(empty, 0.0, (eh * er).sum(1))

    s["pred_len_words"] = [len(h.split()) for h in hyps]
    s["gold_len_words"] = [len(r.split()) for r in refs]

    metric_cols = ["bleu4", "rouge1", "rouge2", "rougeL", "meteor",
                   "bertscore_p", "bertscore_r", "bertscore_f1", "sbert_cosine"]
    summ = {"setting": setting, "n": n, "empty_predictions": int(empty.sum()),
            "corpus_bleu4": sacrebleu.corpus_bleu(hyps, [refs]).score / 100,
            "bertscore_rescaled": rescaled}
    for c in metric_cols:
        lo, hi = _bootstrap_ci(s[c])
        summ[c] = s[c].mean(); summ[f"{c}_std"] = s[c].std(); summ[f"{c}_ci95"] = f"[{lo:.4f}, {hi:.4f}]"
    summ["pred_len_words"] = s.pred_len_words.mean(); summ["gold_len_words"] = s.gold_len_words.mean()

    bar = "=" * 78
    print(f"\n{bar}\nExplanation quality - {setting}  (n={n}, empty={int(empty.sum())})\n{bar}")
    print(f"Corpus BLEU-4 : {summ['corpus_bleu4']:.4f}")
    for c in metric_cols:
        print(f"{c:<14}: {summ[c]:.4f} ± {summ[c + '_std']:.4f}   95% CI {summ[c + '_ci95']}")
    print(f"Avg length (words): pred {summ['pred_len_words']:.1f} | gold {summ['gold_len_words']:.1f}")

    main = ["bleu4", "rougeL", "meteor", "bertscore_f1", "sbert_cosine"]
    by_mech = s.groupby("gold_mechanism")[main].mean().assign(n=s.groupby("gold_mechanism").size())
    by_tgt = s.groupby("gold_target")[main].mean().assign(n=s.groupby("gold_target").size())
    by_pair = (s.groupby(["gold_target", "gold_mechanism"])[main].mean()
                .assign(n=s.groupby(["gold_target", "gold_mechanism"]).size()))
    print("\nBy gold mechanism:\n", by_mech.to_string(float_format=lambda x: f"{x:.4f}"))
    print("\nBy gold target:\n", by_tgt.to_string(float_format=lambda x: f"{x:.4f}"))
    print("\nBy gold target x mechanism:\n", by_pair.to_string(float_format=lambda x: f"{x:.4f}"))

    tag = re.sub(r"[^A-Za-z0-9_-]+", "_", setting)
    s.to_csv(os.path.join(out_dir, f"explanation_scores_per_item_{tag}.csv"), index=False)
    by_mech.to_csv(os.path.join(out_dir, f"explanation_by_mechanism_{tag}.csv"))
    by_tgt.to_csv(os.path.join(out_dir, f"explanation_by_target_{tag}.csv"))
    by_pair.to_csv(os.path.join(out_dir, f"explanation_by_target_x_mechanism_{tag}.csv"))
    return summ, s



p3_summ, p3_scores = evaluate_explanations(pred, CFG["SETTING"], RUN_DIR)


Evaluating 1200 / 1200 memes
Parse status:
 parse_status
json     1188
text       11
regex       1
Generation errors: 0


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Explanation quality - P3-gold-target+mechanism  (n=1200, empty=0)
Corpus BLEU-4 : 0.0391
bleu4         : 0.0409 ± 0.0249   95% CI [0.0395, 0.0424]
rouge1        : 0.3806 ± 0.0571   95% CI [0.3774, 0.3837]
rouge2        : 0.0870 ± 0.0406   95% CI [0.0847, 0.0893]
rougeL        : 0.2101 ± 0.0396   95% CI [0.2080, 0.2121]
meteor        : 0.3078 ± 0.0584   95% CI [0.3044, 0.3110]
bertscore_p   : -0.0006 ± 0.2675   95% CI [-0.0157, 0.0133]
bertscore_r   : 0.2875 ± 0.1043   95% CI [0.2816, 0.2935]
bertscore_f1  : 0.1371 ± 0.1653   95% CI [0.1279, 0.1460]
sbert_cosine  : 0.7200 ± 0.0860   95% CI [0.7151, 0.7247]
Avg length (words): pred 96.7 | gold 65.7

By gold mechanism:
                                     bleu4  rougeL  meteor  bertscore_f1  sbert_cosine    n
gold_mechanism                                                                            
comparative_stereotypical_ridicule 0.0414  0.2093  0.3027        0.1304        0.7390  235
direct_derision                    0.0404  0.2104 

## [Cell 11] Compare P3 with P0 / P1 / P2 (same metric code, same gold explanations)

In [11]:
# Each earlier setting is scored here again, so all four rows come from identical code.
# Paired tests use only the memes present in both settings.
from scipy.stats import wilcoxon


def _find_run_dir(pattern: str):
    if not pattern:
        return None
    hits = [d for d in glob.glob(pattern, recursive=True)
            if os.path.isdir(d) and glob.glob(os.path.join(d, "predictions*.jsonl"))]
    return hits[0] if hits else None


all_summ, all_scores = [p3_summ], {CFG["SETTING"]: p3_scores}
for name, pattern in CFG["PREV_RUNS"].items():
    d = _find_run_dir(pattern)
    if d is None:
        print(f"[skip] {name}: no predictions found for {pattern!r}")
        continue
    print(f"\n>>> {name}: {d}")
    summ, sc = evaluate_explanations(load_predictions(d, ids=set(df_all.id)), name, RUN_DIR)
    all_summ.append(summ); all_scores[name] = sc

expl_summary = pd.DataFrame(all_summ)
order = ["P0-plain", "P1-gold-target", "P2-gold-mechanism", CFG["SETTING"]]
expl_summary["_o"] = expl_summary.setting.map({s: i for i, s in enumerate(order)}).fillna(99)
expl_summary = expl_summary.sort_values("_o").drop(columns="_o").reset_index(drop=True)
expl_summary.to_csv(os.path.join(RUN_DIR, "explanation_summary_all_settings.csv"), index=False)
show = ["setting", "n", "corpus_bleu4", "bleu4", "rouge1", "rouge2", "rougeL", "meteor",
        "bertscore_f1", "sbert_cosine", "pred_len_words"]
print("\nExplanation quality across settings:")
print(expl_summary[show].to_string(index=False, float_format=lambda x: f"{x:.4f}"))


def paired_test(a: pd.DataFrame, b: pd.DataFrame, metric: str, n_boot=2000, seed=42):
    m = a[["id", metric]].merge(b[["id", metric]], on="id", suffixes=("_a", "_b"))
    diff = (m[f"{metric}_a"] - m[f"{metric}_b"]).to_numpy()
    rng = np.random.default_rng(seed)
    boots = rng.choice(diff, size=(n_boot, len(diff)), replace=True).mean(1)
    try:
        pval = wilcoxon(diff).pvalue if np.any(diff != 0) else 1.0
    except ValueError:
        pval = float("nan")
    return {"n_paired": len(diff), "mean_diff": diff.mean(),
            "ci95_low": np.percentile(boots, 2.5), "ci95_high": np.percentile(boots, 97.5),
            "wilcoxon_p": pval}


tests = []
for name, sc in all_scores.items():
    if name == CFG["SETTING"]:
        continue
    for metric in ["bleu4", "rougeL", "meteor", "bertscore_f1", "sbert_cosine"]:
        tests.append({"comparison": f"{CFG['SETTING']} - {name}", "metric": metric,
                      **paired_test(p3_scores, sc, metric)})
if tests:
    tests = pd.DataFrame(tests)
    tests["significant_0.05"] = tests.wilcoxon_p < 0.05
    print("\nPaired differences (positive = P3 better), bootstrap 95% CI + Wilcoxon signed-rank:")
    print(tests.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    tests.to_csv(os.path.join(RUN_DIR, "explanation_paired_tests.csv"), index=False)
else:
    print("\nNo earlier settings found - add the P0/P1/P2 outputs as inputs to compare.")


[skip] P0-plain: no predictions found for '/kaggle/input/**/qwen35_9b_P0_plain'
[skip] P1-gold-target: no predictions found for '/kaggle/input/**/qwen35_9b_P1_gold_target'
[skip] P2-gold-mechanism: no predictions found for '/kaggle/input/**/qwen35_9b_P2_gold_mechanism'

Explanation quality across settings:
                 setting    n  corpus_bleu4  bleu4  rouge1  rouge2  rougeL  meteor  bertscore_f1  sbert_cosine  pred_len_words
P3-gold-target+mechanism 1200        0.0391 0.0409  0.3806  0.0870  0.2101  0.3078        0.1371        0.7200         96.6642

No earlier settings found - add the P0/P1/P2 outputs as inputs to compare.


## [Cell 12] Summary tables (CSV + LaTeX) and saved artefacts

In [12]:
cols = ["bleu4", "rouge1", "rouge2", "rougeL", "meteor", "bertscore_f1", "sbert_cosine"]
tab = expl_summary.set_index("setting")[cols]
tab.insert(0, "n", expl_summary.set_index("setting")["n"])
with open(os.path.join(RUN_DIR, "summary_tables.tex"), "w") as f:
    f.write(tab.to_latex(float_format="%.4f",
                         caption=f"{CFG['MODEL_ID']}: explanation quality under P0-P3 prompting",
                         label="tab:explanation_all_settings"))
    if isinstance(tests, pd.DataFrame) and len(tests):
        f.write("\n")
        f.write(tests.set_index(["comparison", "metric"])
                .to_latex(float_format="%.4f", caption="Paired differences in explanation scores",
                          label="tab:explanation_paired"))

pred.to_csv(os.path.join(RUN_DIR, "predictions_parsed.csv"), index=False)
with open(os.path.join(RUN_DIR, "run_config.json"), "w") as f:
    json.dump({**CFG, "prompt_example": build_prompt_p3("individual", "direct_derision",
                                                        CFG["EXPLANATION_LANGUAGE"])},
              f, indent=2, ensure_ascii=False)
print("\nSaved to", RUN_DIR, ":", sorted(os.listdir(RUN_DIR)))



Saved to /kaggle/working/results/qwen35_9b_P3_gold_target_mechanism : ['explanation_by_mechanism_P3-gold-target_mechanism.csv', 'explanation_by_target_P3-gold-target_mechanism.csv', 'explanation_by_target_x_mechanism_P3-gold-target_mechanism.csv', 'explanation_scores_per_item_P3-gold-target_mechanism.csv', 'explanation_summary_all_settings.csv', 'predictions_parsed.csv', 'predictions_shard0.jsonl', 'run_config.json', 'summary_tables.tex']
